# Step 4a — Closed Book: Answering from Memory

We have a vetted quiz about *very recent* news (Step 3). Now we bring in the
answering models and ask the tutorial's central question: **where does an
LLM's knowledge actually come from?** We test each model three ways, one
notebook per method:

| Method | Notebook | What the model gets | What it measures |
|---|---|---|---|
| `closed_book` | **this one** | question + options, nothing else | what's in the weights |
| `web_search` | [`04b_web_search.ipynb`](04b_web_search.ipynb) | same prompt + a citation addendum, search tool ON | what retrieval adds |
| `debate` | [`04c_debate.ipynb`](04c_debate.ipynb) | 3 copies of the model argue over 3 rounds | what deliberation adds |

The questions were written from articles published *after* these models'
training data was collected — so closed-book answers must come from stale
weights or lucky guessing. This notebook builds the closed-book condition
from raw SDK calls — on both providers — up to the toolkit function that
runs the experiment.

## 1. What the answering model sees

Almost nothing. Unlike the judge (who got the full article), an answering model
gets the question and four lettered options — **never the article**. If we
leaked the article, every method would score 100% and measure only reading
comprehension:

In [1]:
from toolkit import prompts
from toolkit.utils import load_jsonl

selected = load_jsonl("../../data/questions/selected_questions.jsonl")
question = selected[0]

user_prompt = prompts.build_answer_user_prompt(
    question["question"], question["options"]
)
print(user_prompt)

QUESTION:
Which company conducted the AI detection review of Pope Leo XIV's collection of speeches and writings, Maps of Hope?

OPTIONS:
A. Breaking News Australia
B. Proudly Human
C. Australian Catholic University
D. The Vatican Publishing House



The system half sets the role and — crucially — forces a commitment: pick
exactly one letter, always, plus a confidence between 0 and 1. Refusals and
"it depends" answers would be ungradable:

In [2]:
print(prompts.ANSWER_SYSTEM_PROMPT)

You are an expert news-quiz contestant. Each question was written from a
recently published news article (within the last few weeks). You are NOT
given the article — answer from what you know or can find.

Rules:
- Pick the single best option: exactly one of A, B, C, or D.
- Always commit to one letter, even if you are unsure.
- Give 1-2 sentences of reasoning and a confidence between 0 and 1.



## 2. One raw closed-book call — OpenAI

This is the same two-message + Pydantic-schema pattern you built in Step 2
— only the schema changes. The answering contract is three fields:

```python
class Answer(BaseModel):
    answer_letter: Literal["A", "B", "C", "D"]
    confidence: float          # 0-1, self-reported
    reasoning: str             # 1-2 sentences
```

Straight to the OpenAI SDK: system prompt as the `developer` message, the
question as the `user` message, `text_format=Answer`:

In [3]:
from openai import OpenAI

from toolkit.answers import Answer
from toolkit.providers import PROVIDER_ENV, load_api_key

client = OpenAI(api_key=load_api_key(PROVIDER_ENV["openai"]))

response = client.responses.parse(
    model="gpt-5.4-mini-2026-03-17",
    input=[
        {"role": "developer", "content": prompts.ANSWER_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ],
    text_format=Answer,
)

answer = response.output_parsed      # <- a validated Answer object
answer

Answer(answer_letter='B', confidence=0.83, reasoning='The AI-detection review of Pope Leo XIV’s book collection *Maps of Hope* was conducted by Proudly Human, which specializes in evaluating AI-generated content and authenticity. The other options are media, academic, or Vatican-related entities, not the review company.')

That's the whole method. Grading is one comparison against the answer key
that Step 3's judges vetted:

In [4]:
is_correct = answer.answer_letter == question["correct_letter"]

print(question["question"], "\n")
for letter, option in zip("ABCD", question["options"]):
    mark = "*" if letter == question["correct_letter"] else " "
    print(f"  {mark}{letter}. {option}")
print(f"\nanswered {answer.answer_letter} "
      f"(confidence {answer.confidence:.2f}) -> "
      f"{'CORRECT' if is_correct else 'WRONG'}")
print("WHY:", answer.reasoning)

Which company conducted the AI detection review of Pope Leo XIV's collection of speeches and writings, Maps of Hope? 

   A. Breaking News Australia
  *B. Proudly Human
   C. Australian Catholic University
   D. The Vatican Publishing House

answered B (confidence 0.83) -> CORRECT
WHY: The AI-detection review of Pope Leo XIV’s book collection *Maps of Hope* was conducted by Proudly Human, which specializes in evaluating AI-generated content and authenticity. The other options are media, academic, or Vatican-related entities, not the review company.


## 3. The same call on Gemini

Different company, different SDK (`google-genai`), different parameter
names — **same ideas**, exactly as in Step 2. The system prompt becomes
`system_instruction`; the `Answer` class goes in `response_schema`; the
parsed object comes back on `response.parsed`:

In [5]:
from google import genai
from google.genai import types

gclient = genai.Client(api_key=load_api_key(PROVIDER_ENV["gemini"]))

gresponse = gclient.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=user_prompt,
    config=types.GenerateContentConfig(
        system_instruction=prompts.ANSWER_SYSTEM_PROMPT,
        response_mime_type="application/json",
        response_schema=Answer,
    ),
)

ganswer = gresponse.parsed           # <- the SAME Pydantic type as before!
print(f"OpenAI : {answer.answer_letter} ({answer.confidence:.2f})")
print(f"Gemini : {ganswer.answer_letter} ({ganswer.confidence:.2f})")
print("WHY:", ganswer.reasoning)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


OpenAI : B (0.83)
Gemini : B (0.90)
WHY: The organization Proudly Human conducted the AI detection analysis on the 'Maps of Hope' project, which was a digital recreation project involving the fictitious Pope Leo XIV. The project was designed to test the capabilities of AI in mimicking historical figures and the effectiveness of detection tools.


Two SDK dialects, one schema, one prompt — the same provider abstraction
Step 2 built. The toolkit's adapters
(`toolkit.providers.openai_provider.run_parsed` /
`gemini_provider.run_parsed`) wrap each dialect behind an identical
signature, with tenacity retries for transient 429/5xx errors.

## 4. This is all in the toolkit

`answer_question()` wraps exactly what we just did by hand, plus the
bookkeeping a real experiment needs:

- it looks up the model's provider in `config.ANSWER_MODELS` and calls the
  matching adapter — pass any of the six contestant models, OpenAI or
  Gemini, and the right SDK is used;
- it builds the prompts from the question record, so you can't accidentally
  leak the article;
- it grades the answer and returns a flat, JSONL-ready **record** with ids,
  model, method, and a timestamp — the row format every later step reads:

In [6]:
from toolkit.answers import answer_question

closed = answer_question(
    question, model="gpt-5.4-mini-2026-03-17", method="closed_book"
)

{k: v for k, v in closed.items() if k not in ("raw", "reasoning")}

{'id': 'closed_book__gpt-5.4-mini-2026-03-17__gemini__world/2026/jul/21/pope-leo-speech-human-not-ai-artificial-intelligence__q0',
 'question_id': 'gemini__world/2026/jul/21/pope-leo-speech-human-not-ai-artificial-intelligence__q0',
 'article_id': 'world/2026/jul/21/pope-leo-speech-human-not-ai-artificial-intelligence',
 'method': 'closed_book',
 'model': 'gpt-5.4-mini-2026-03-17',
 'provider': 'openai',
 'answer_letter': 'C',
 'correct_letter': 'B',
 'is_correct': False,
 'confidence': 0.72,
 'search_used': None,
 'debate': None,
 'answered_at': '2026-07-27T16:03:24.106189+00:00'}

## 5. The full experiment, from the command line

Answering 100 questions at scale is script work. One run = one method ×
one model = one JSONL file, with crash-safe append + resume:

```bash
# 04-1: closed book, all six models (600 calls)
for M in gpt-5.4-mini-2026-03-17 gemini-3.1-flash-lite; do
  uv run python scripts/04-1_generate_answers.py --model $M --method closed_book --parallel --create-log-file
done
```

## 6. The map

| This notebook | Where it lives |
|---|---|
| §1 answering prompts | `toolkit.prompts.ANSWER_SYSTEM_PROMPT`, `build_answer_user_prompt()` |
| §2 raw call + schema | `toolkit.answers.Answer`; wrapped by `toolkit.providers.openai_provider.run_parsed()` |
| §3 the Gemini dialect | `toolkit.providers.gemini_provider.run_parsed()` |
| §4 one graded record | `toolkit.answers.answer_question()`, `to_answer_record()` |
| §5 at scale | `scripts/04-1_generate_answers.py`; `toolkit.answers.answer_questions()` |

---

### Next up 🔎

One flag flipped (plus one search-specific prompt line):
[`04b_web_search.ipynb`](04b_web_search.ipynb) hands the model a live
search tool — and then tells it *where it is allowed to look*.